# ViT5-based AI-Generated Vietnamese Text Detection

Binary classification model to detect AI-generated Vietnamese text using vietai/ViT5.

**Labels:**
- 0 = AI-generated text
- 1 = Human-written text

## 1. Setup & Imports

In [1]:
# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================
!pip install -q transformers sentencepiece torch pandas numpy scikit-learn matplotlib seaborn tqdm unicodedata2

In [2]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================
import os
import re
import unicodedata
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

/home/hhbach/anaconda3/envs/aigenDetection_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using device: cuda
   GPU: NVIDIA RTX A4000
   Memory: 15.6 GB


## 2. Configuration

In [3]:
# ============================================================
# 3. CONFIGURATION
# ============================================================
class Config:
    # Paths
    DATASET_PATH = "/home/hhbach/nlp_aigen_detection/kd_test_data"
    MODEL_SAVE_PATH = "/home/hhbach/nlp_aigen_detection/best_model_vit5.pt"
    
    # Model
    MODEL_NAME = "VietAI/vit5-base"  # Or VietAI/vit5-large
    MAX_LENGTH = 256
    
    # Training
    BATCH_SIZE = 16
    EPOCHS = 5
    LEARNING_RATE = 1e-4  # T5 needs higher LR than BERT
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    
    # Split ratios
    TEST_SIZE = 0.15
    VAL_SIZE = 0.15  # of remaining after test split
    
    # Labels
    LABEL_AI = 0
    LABEL_HUMAN = 1
    LABEL_TOKENS = {0: "AI", 1: "HUMAN"}

print("📋 Configuration:")
for attr in dir(Config):
    if not attr.startswith('_'):
        print(f"   {attr}: {getattr(Config, attr)}")

📋 Configuration:
   BATCH_SIZE: 16
   DATASET_PATH: /home/hhbach/nlp_aigen_detection/kd_test_data
   EPOCHS: 5
   LABEL_AI: 0
   LABEL_HUMAN: 1
   LABEL_TOKENS: {0: 'AI', 1: 'HUMAN'}
   LEARNING_RATE: 0.0001
   MAX_LENGTH: 256
   MODEL_NAME: VietAI/vit5-base
   MODEL_SAVE_PATH: /home/hhbach/nlp_aigen_detection/best_model_vit5.pt
   TEST_SIZE: 0.15
   VAL_SIZE: 0.15
   WARMUP_RATIO: 0.1
   WEIGHT_DECAY: 0.01


## 3. Data Loading

In [4]:
# ============================================================
# 4. DATA LOADING FUNCTIONS
# ============================================================

def read_text_file(filepath):
    encodings = ['utf-8', 'utf-16-le', 'utf-16-be', 'utf-16', 'latin-1', 'cp1252']
    for enc in encodings:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                content = f.read()
                content = content.replace('\x00', '')
                return content
        except:
            continue
    try:
        with open(filepath, 'rb') as f:
            content = f.read()
            content = content.decode('utf-8', errors='replace').replace('\x00', '')
            return content
    except:
        return ""

def extract_group_id(filename, label):
    match = re.search(r'\((\d+)\)', filename)
    if match:
        article_num = match.group(1)
        prefix = "AI" if label == 0 else "Human"
        return f"{prefix}_article_{article_num}"
    else:
        prefix = "AI" if label == 0 else "Human"
        return f"{prefix}_{filename.replace('.txt', '')}"

def load_dataset(base_path):
    data = []
    ai_path = os.path.join(base_path, "AI")
    for category in os.listdir(ai_path):
        cat_path = os.path.join(ai_path, category)
        if os.path.isdir(cat_path):
            for filename in os.listdir(cat_path):
                if filename.endswith('.txt'):
                    filepath = os.path.join(cat_path, filename)
                    text = read_text_file(filepath)
                    if text.strip():
                        parts = filename.split('_')
                        source = parts[1] if len(parts) > 1 else 'Unknown'
                        group_id = extract_group_id(filename, Config.LABEL_AI)
                        data.append({
                            'filepath': filepath,
                            'filename': filename,
                            'text': text,
                            'label': Config.LABEL_AI,
                            'category': category,
                            'source': source,
                            'group_id': group_id
                        })
    human_path = os.path.join(base_path, "Human")
    for category in os.listdir(human_path):
        cat_path = os.path.join(human_path, category)
        if os.path.isdir(cat_path):
            for filename in os.listdir(cat_path):
                if filename.endswith('.txt'):
                    filepath = os.path.join(cat_path, filename)
                    text = read_text_file(filepath)
                    if text.strip():
                        parts = filename.split('_')
                        source = parts[1] if len(parts) > 1 else 'Unknown'
                        group_id = extract_group_id(filename, Config.LABEL_HUMAN)
                        data.append({
                            'filepath': filepath,
                            'filename': filename,
                            'text': text,
                            'label': Config.LABEL_HUMAN,
                            'category': category,
                            'source': source,
                            'group_id': group_id
                        })
    df = pd.DataFrame(data)
    return df

print("🔍 Loading dataset...")
df = load_dataset(Config.DATASET_PATH)
print(f"✅ Loaded {len(df)} samples")
print(f"\n📊 Label distribution:")
print(df['label'].value_counts().rename({0: 'AI', 1: 'Human'}))
print(f"\n📊 Total unique groups: {df['group_id'].nunique()}")

🔍 Loading dataset...
✅ Loaded 39974 samples

📊 Label distribution:
label
Human    25065
AI       14909
Name: count, dtype: int64

📊 Total unique groups: 15765


## 4. Preprocessing

In [5]:
# ============================================================
# 5. PREPROCESSING
# ============================================================

def preprocess_text(text):
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text)
    text = ''.join(char for char in text if char == '\n' or unicodedata.category(char)[0] != 'C')
    text = text.lower()
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(line for line in lines if line)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

print("🧹 Preprocessing texts...")
tqdm.pandas()
df['text_processed'] = df['text'].progress_apply(preprocess_text)
initial_len = len(df)
df = df[df['text_processed'].str.len() > 0].reset_index(drop=True)
print(f"✅ Preprocessing complete: {initial_len - len(df)} empty texts removed")
print(f"   Final dataset size: {len(df)}")

🧹 Preprocessing texts...


100%|██████████| 39974/39974 [00:37<00:00, 1058.75it/s]

✅ Preprocessing complete: 0 empty texts removed
   Final dataset size: 39974


## 5. Group-Based Split

In [6]:
# ============================================================
# 6. GROUP-BASED TRAIN/VAL/TEST SPLIT
# ============================================================

def group_based_split(df, test_size=0.15, val_size=0.15, random_state=42):
    print(f"📊 Total groups: {df['group_id'].nunique()}")
    print(f"📊 Total samples: {len(df)}")
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(gss_test.split(df, groups=df['group_id']))
    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    adjusted_val_size = val_size / (1 - test_size)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=adjusted_val_size, random_state=random_state)
    train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['group_id']))
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, test_df

print("✂️ Splitting data by groups (70/15/15)...")
print("="*60)
train_df, val_df, test_df = group_based_split(df, test_size=Config.TEST_SIZE, val_size=Config.VAL_SIZE)
print(f"\n📊 Split sizes:")
print(f"   Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%) - {train_df['group_id'].nunique()} groups")
print(f"   Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%) - {val_df['group_id'].nunique()} groups")
print(f"   Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%) - {test_df['group_id'].nunique()} groups")
train_groups = set(train_df['group_id'].unique())
val_groups = set(val_df['group_id'].unique())
test_groups = set(test_df['group_id'].unique())
print(f"\n🔍 Group overlap check:")
print(f"   Train ∩ Val: {len(train_groups & val_groups)} groups (should be 0)")
print(f"   Train ∩ Test: {len(train_groups & test_groups)} groups (should be 0)")
print(f"   Val ∩ Test: {len(val_groups & test_groups)} groups (should be 0)")

✂️ Splitting data by groups (70/15/15)...
📊 Total groups: 15765
📊 Total samples: 39974

📊 Split sizes:
   Train: 27934 (69.9%) - 11035 groups
   Validation: 6088 (15.2%) - 2365 groups
   Test: 5952 (14.9%) - 2365 groups

🔍 Group overlap check:
   Train ∩ Val: 0 groups (should be 0)
   Train ∩ Test: 0 groups (should be 0)
   Val ∩ Test: 0 groups (should be 0)


## 6. ViT5 Tokenization

In [7]:
# ============================================================
# 7. TOKENIZATION
# ============================================================

print("🔤 Loading ViT5 tokenizer...")

# For ViT5, we use AutoTokenizer which handles sentencepiece properly
tokenizer = AutoTokenizer.from_pretrained(
    Config.MODEL_NAME,
    use_fast=False
)

print(f"✅ Loaded tokenizer: {Config.MODEL_NAME}")
sample_text = train_df.iloc[0]['text_processed'][:100]
input_text = f"detect ai text: {sample_text}"
sample_tokens = tokenizer(
    input_text,
    max_length=Config.MAX_LENGTH,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)
print(f"\n📝 Tokenization example:")
print(f"   Input text: {input_text[:50]}...")
print(f"   Token count: {sample_tokens['input_ids'].shape[1]}")
print(f"   Input IDs shape: {sample_tokens['input_ids'].shape}")

🔤 Loading ViT5 tokenizer...


KeyError: 0

## 7. Dataset Class

In [ ]:
# ============================================================
# 8. CUSTOM DATASET CLASS
# ============================================================

class ViT5DetectionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df['text_processed'].tolist()
        self.labels = df['label'].tolist()
        self.group_ids = df['group_id'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        input_text = f"detect ai text: {text}"
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long),
            'group_id': self.group_ids[idx]
        }

print("📦 Creating datasets...")
train_dataset = ViT5DetectionDataset(train_df, tokenizer, Config.MAX_LENGTH)
val_dataset = ViT5DetectionDataset(val_df, tokenizer, Config.MAX_LENGTH)
test_dataset = ViT5DetectionDataset(test_df, tokenizer, Config.MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
print(f"✅ DataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 8. Model Architecture

In [ ]:
# ============================================================
# 9. MODEL ARCHITECTURE
# ============================================================

class ViT5Classifier(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout_rate=0.1):
        super(ViT5Classifier, self).__init__()
        # Load the base T5 model and use encoder only
        self.t5 = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.t5.config.d_model
        self.dropout = nn.Dropout(dropout_rate)
        self.layer_norm = nn.LayerNorm(self.hidden_size)
        self.classifier = nn.Linear(self.hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = encoder_outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_hidden = torch.sum(hidden_states * mask_expanded, dim=1)
        mean_pooled = sum_hidden / torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = self.layer_norm(mean_pooled)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

print("🤖 Initializing ViT5 model...")
model = ViT5Classifier(Config.MODEL_NAME, num_classes=2)
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model initialized")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   T5 d_model: {model.hidden_size}")

## 9. Training Setup

In [ ]:
# ============================================================
# 10. TRAINING SETUP
# ============================================================

n_samples = len(train_df)
n_ai = len(train_df[train_df['label'] == 0])
n_human = len(train_df[train_df['label'] == 1])
weight_ai = n_samples / (2 * n_ai)
weight_human = n_samples / (2 * n_human)
class_weights = torch.tensor([weight_ai, weight_human], dtype=torch.float).to(device)
print(f"📊 Class weights: AI={weight_ai:.4f}, Human={weight_human:.4f}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
t5_params = list(model.t5.named_parameters())
classifier_params = list(model.classifier.named_parameters()) + list(model.layer_norm.named_parameters())
optimizer_grouped_parameters = [
    {'params': [p for n, p in t5_params], 'lr': Config.LEARNING_RATE},
    {'params': [p for n, p in classifier_params], 'lr': Config.LEARNING_RATE * 10}
]
optimizer = AdamW(optimizer_grouped_parameters, weight_decay=Config.WEIGHT_DECAY)
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * Config.EPOCHS
warmup_steps = int(total_steps * Config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
print(f"\n⚙️ Training config: steps={total_steps}, warmup={warmup_steps}")
print(f"📚 LR: T5={Config.LEARNING_RATE}, Classifier={Config.LEARNING_RATE * 10}")

In [ ]:
# ============================================================
# 11. TRAINING FUNCTIONS
# ============================================================

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    total_grad_norm = 0
    progress_bar = tqdm(dataloader, desc="Training")
    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        total_grad_norm += grad_norm.item()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        if batch_idx % 50 == 0:
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'grad_norm': f'{grad_norm.item():.2f}'})
    avg_loss = total_loss / len(dataloader)
    avg_grad_norm = total_grad_norm / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, avg_grad_norm

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'ai_f1': f1_score(all_labels, all_preds, average='binary', pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5,
        'predictions': all_preds,
        'labels': all_labels,
        'probs': all_probs
    }
    return metrics

## 10. Training Loop

In [ ]:
# ============================================================
# 12. TRAINING LOOP
# ============================================================

history = {
    'train_loss': [],
    'train_acc': [],
    'train_grad_norm': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': [],
    'val_macro_f1': [],
    'val_ai_f1': []
}
best_val_macro_f1 = 0
best_model_state = None
print("\n🚀 Starting ViT5 training...")
print("="*60)
for epoch in range(Config.EPOCHS):
    print(f"\n📌 Epoch {epoch + 1}/{Config.EPOCHS}")
    print("-"*60)
    train_loss, train_acc, train_grad_norm = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_grad_norm'].append(train_grad_norm)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_macro_f1'].append(val_metrics['macro_f1'])
    history['val_ai_f1'].append(val_metrics['ai_f1'])
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Grad Norm: {train_grad_norm:.2f}")
    print(f"   Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
    print(f"   Val F1(weighted): {val_metrics['f1']:.4f} | Val F1(macro): {val_metrics['macro_f1']:.4f} | Val AI-F1: {val_metrics['ai_f1']:.4f}")
    print(f"   Val ROC-AUC: {val_metrics['roc_auc']:.4f}")
    if val_metrics['macro_f1'] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics['macro_f1']
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"   🌟 New best model saved! (Val Macro F1: {best_val_macro_f1:.4f})")
print("\n" + "="*60)
print(f"✅ Training complete! Best Val Macro F1: {best_val_macro_f1:.4f}")

## 11. Save Best Model

In [ ]:
# ============================================================
# 13. SAVE BEST MODEL
# ============================================================

if best_model_state is not None:
    checkpoint = {
        'model_state_dict': best_model_state,
        'config': {
            'model_name': Config.MODEL_NAME,
            'max_length': Config.MAX_LENGTH,
            'hidden_size': model.hidden_size
        },
        'best_val_macro_f1': best_val_macro_f1
    }
    torch.save(checkpoint, Config.MODEL_SAVE_PATH)
    print(f"💾 Best model saved to: {Config.MODEL_SAVE_PATH}")
    model.load_state_dict(best_model_state)
else:
    print("⚠️ No best model found, using last epoch")

## 12. Test Evaluation

In [ ]:
# ============================================================
# 14. TEST SET EVALUATION
# ============================================================

print("\n" + "="*60)
print("📊 TEST SET EVALUATION")
print("="*60)
test_metrics = evaluate(model, test_loader, criterion, device)
print(f"\n📈 Test Metrics:")
print(f"   Loss: {test_metrics['loss']:.4f}")
print(f"   Accuracy: {test_metrics['accuracy']:.4f}")
print(f"   Precision: {test_metrics['precision']:.4f}")
print(f"   Recall: {test_metrics['recall']:.4f}")
print(f"   F1-Score (weighted): {test_metrics['f1']:.4f}")
print(f"   F1-Score (macro): {test_metrics['macro_f1']:.4f}")
print(f"   AI F1-Score: {test_metrics['ai_f1']:.4f}")
print(f"   ROC-AUC: {test_metrics['roc_auc']:.4f}")
print("\n📋 Classification Report:")
print(classification_report(test_metrics['labels'], test_metrics['predictions'], target_names=['AI', 'Human'], digits=4))

## 13. Visualizations

In [ ]:
# ============================================================
# 15. VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0, 0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0, 0].set_title('Loss Curves')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[0, 1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[0, 1].set_title('Accuracy Curves')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['val_macro_f1'], label='Val Macro F1', marker='s', color='green')
axes[1, 0].plot(history['val_ai_f1'], label='Val AI F1', marker='^', color='orange')
axes[1, 0].set_title('Validation F1 Scores')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('F1 Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

cm = confusion_matrix(test_metrics['labels'], test_metrics['predictions'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1], xticklabels=['AI', 'Human'], yticklabels=['AI', 'Human'])
axes[1, 1].set_title('Confusion Matrix (Test Set)')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 14. Sample Inference

In [ ]:
# ============================================================
# 16. SAMPLE INFERENCE
# ============================================================

def predict_single_text(model, tokenizer, text, device, max_length=256):
    model.eval()
    text = preprocess_text(text)
    input_text = f"detect ai text: {text}"
    encoding = tokenizer(input_text, max_length=max_length, padding='max_length', truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(logits, dim=1).item()
    return {
        'prediction': 'AI' if pred == 0 else 'Human',
        'p_ai': probs[0, 0].item(),
        'p_human': probs[0, 1].item(),
        'confidence': max(probs[0, 0].item(), probs[0, 1].item())
    }

print("\n📝 Sample Predictions:")
print("="*60)
for i in range(5):
    idx = random.randint(0, len(test_df) - 1)
    sample = test_df.iloc[idx]
    true_label = 'AI' if sample['label'] == 0 else 'Human'
    result = predict_single_text(model, tokenizer, sample['text'], device)
    print(f"\nSample {i+1}:")
    print(f"   Text: {sample['text_processed'][:100]}...")
    print(f"   True: {true_label} | Predicted: {result['prediction']} (conf: {result['confidence']:.2%})")
    print(f"   P(AI)={result['p_ai']:.3f}, P(Human)={result['p_human']:.3f}")

# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "="*60)
print("🎯 TRAINING SUMMARY")
print("="*60)
print(f"\n📊 Model: {Config.MODEL_NAME}")
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Best validation Macro F1: {best_val_macro_f1:.4f}")
print(f"📊 Test Weighted F1: {test_metrics['f1']:.4f}")
print(f"📊 Test Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"📊 Test AI F1: {test_metrics['ai_f1']:.4f}")
print(f"📊 Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"\n✅ Model saved to: {Config.MODEL_SAVE_PATH}")
print("\n📝 Key Differences from PhoBERT:")
print("   1. T5 architecture: Encoder-only usage for classification")
print("   2. Task prefix: 'detect ai text:' required for T5")
print("   3. Mean pooling instead of [CLS] token")
print("   4. Higher learning rate (1e-4 vs 2e-5)")
print("   5. SentencePiece tokenizer required")
print("\n🚀 Next steps:")
print("   - Compare performance with PhoBERT")
print("   - Try ensemble: PhoBERT + ViT5")
print("   - Experiment with freezing T5 layers")

In [ ]:
# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "="*60)
print("🎯 TRAINING SUMMARY")
print("="*60)
print(f"\n📊 Model: {Config.MODEL_NAME}")
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Best validation F1: {best_val_f1:.4f}")
print(f"📊 Test F1: {test_metrics['f1']:.4f}")
print(f"📊 Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"\n✅ Model saved to: {Config.MODEL_SAVE_PATH}")
print("\n📝 Key Differences from PhoBERT:")
print("   1. T5 architecture: Encoder-Decoder vs Encoder-only")
print("   2. Task prefix: 'detect ai text:' required for T5")
print("   3. Mean pooling instead of [CLS] token")
print("   4. Higher learning rate (1e-4 vs 2e-5)")
print("   5. More parameters (~250M vs ~135M)")
print("\n🚀 Next steps:")
print("   - Compare performance with PhoBERT")
print("   - Try ensemble: PhoBERT + ViT5")
print("   - Experiment with freezing T5 layers")